# Text-to-SQL: Colab GPU runner

Runs the GPU side of the project: (a) zero-shot baseline predictions, (b) a 2,000-example QLoRA
smoke-test training run, (c) predictions with the trained adapter. Predictions are scored here for a
quick look, and the `.jsonl` files are downloaded at the end so you can score them locally on CPU with
`scripts/evaluate.py`.

**Before you start**
- Runtime → Change runtime type → **GPU** (T4 is fine; L4/A100 is faster).
- The cleaned dataset must be on the Hub: `python scripts/prepare_data.py --hub-repo <hf-user>/sql-create-context-clean`.
- Secrets: add these in the **Secrets** panel (key icon in the left sidebar) and switch on
  *Notebook access* for each. Any that are missing are asked for with `getpass`. Nothing is
  stored in the notebook. Names match `.env.example`:

  | Name | Needed for | Scope |
  |---|---|---|
  | `HF_TOKEN` | private dataset download | Read |
  | `WANDB_API_KEY` | training logs | https://wandb.ai/authorize |
  | `GH_COLAB_TOKEN` | *optional*, cloning only if the repo is private | fine-grained, this repo, Contents: Read-only |

## 0. Settings

In [ ]:
GITHUB_USER = "NaveenVikramR"
GITHUB_REPO = "TextSqlModel"
GITHUB_BRANCH = "main"
HF_DATASET = None   # None -> "<your HF username>/sql-create-context-clean", resolved after HF login
BASE_MODEL = "unsloth/Llama-3.2-3B-Instruct"
CONFIG = "configs/qlora_llama32_3b.yaml"
SMOKE_SAMPLES = 2000
SMOKE_DIR = "outputs/smoke-llama32-3b"

from getpass import getpass

def get_secret(name, prompt=None):
    """Colab Secrets first, then an interactive getpass prompt. Never prints the value."""
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            print(f"{name}: loaded from Colab Secrets")
            return value.strip()
    except Exception:  # not on Colab, secret missing, or notebook access not granted
        pass
    return getpass(prompt or f"{name}: ").strip()

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU"
BF16 = torch.cuda.is_bf16_supported()
print("GPU:", torch.cuda.get_device_name(0), "| bf16 supported:", BF16)

## 2. Clone the repo
Uses the `GH_COLAB_TOKEN` secret if set; otherwise press Enter at the prompt for the public repo. A token is used only for the clone and then removed from `.git/config`.

In [ ]:
import os, subprocess

REPO_DIR = f"/content/{GITHUB_REPO}"
if not os.path.isdir(REPO_DIR):
    token = get_secret("GH_COLAB_TOKEN", "GitHub token (press Enter to skip for a public repo): ")
    auth = f"x-access-token:{token}@" if token else ""
    url = f"https://{auth}github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
    proc = subprocess.run(["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, url, REPO_DIR],
                          capture_output=True, text=True)
    if proc.returncode != 0:
        err = proc.stderr.replace(token, "***") if token else proc.stderr
        raise RuntimeError("git clone failed:\n" + err)
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin",
                    f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"], check=True)
    del token, url
else:
    print("Repo already cloned; delete", REPO_DIR, "to re-clone a newer commit.")

%cd {REPO_DIR}
!git log --oneline -1

## 3. Install training dependencies
Colab already ships a CUDA build of PyTorch, so it is not reinstalled. If pip asks you to restart the session, do so and re-run cells 0, 1 and `%cd` before continuing.

In [ ]:
!pip install -q -r requirements-train.txt
!pip list 2>/dev/null | grep -iE "^(torch|transformers|trl|peft|bitsandbytes|datasets|wandb) "

## 4. Log in to Hugging Face and Weights & Biases
Tokens come from the `HF_TOKEN` / `WANDB_API_KEY` secrets (or a prompt). They are also exported to the environment so the `!python scripts/...` subprocesses pick them up.

In [ ]:
from huggingface_hub import login, whoami
hf_token = get_secret("HF_TOKEN", "Hugging Face token: ")
login(token=hf_token, add_to_git_credential=False)
os.environ["HF_TOKEN"] = hf_token
HF_USER = whoami()["name"]
if not HF_DATASET:
    HF_DATASET = f"{HF_USER}/sql-create-context-clean"
print("HF user:", HF_USER, "| dataset:", HF_DATASET)

import wandb
wandb_key = get_secret("WANDB_API_KEY", "W&B API key: ")
wandb.login(key=wandb_key)
os.environ["WANDB_API_KEY"] = wandb_key
del hf_token, wandb_key

## 5. Precision settings
The YAML targets an L4 (bf16). On a T4 (no bf16) these overrides switch to fp16 and halve the per-device batch, keeping the effective batch at 16.

In [ ]:
PRECISION = "" if BF16 else " ".join([
    "--set quantization.bnb_4bit_compute_dtype=float16",
    "--set training.bf16=false",
    "--set training.fp16=true",
    "--set training.per_device_train_batch_size=8",
    "--set training.per_device_eval_batch_size=8",
    "--set training.gradient_accumulation_steps=2",
])
print(PRECISION or "bf16 GPU: using the YAML as-is")

## (a) Baseline: zero-shot predictions from the base model on the test split

In [ ]:
!python scripts/generate.py --base-model {BASE_MODEL} --dataset {HF_DATASET} --split test \
    --output outputs/preds_base.jsonl --batch-size 16
!python scripts/evaluate.py --predictions outputs/preds_base.jsonl --report outputs/report_base.json

## (b) Smoke-test training: 2000 examples
Verifies the full training path (data, 4-bit load, LoRA, W&B logging, adapter save). Evaluates and checkpoints every 25 steps (~125 steps total).

In [ ]:
!python scripts/train.py --config {CONFIG} \
    --max-train-samples {SMOKE_SAMPLES} \
    --output-dir {SMOKE_DIR} \
    --set data.dataset={HF_DATASET} \
    --set data.max_eval_samples=200 \
    --set run_name=smoke-llama32-3b \
    --set training.logging_steps=5 \
    --set training.eval_steps=25 \
    --set training.save_steps=25 \
    {PRECISION}
!ls -la {SMOKE_DIR}/final

## (c) Predictions with the trained adapter
The base model is loaded in 16-bit and the LoRA adapter applied on top (the same setup as the later merge → GGUF step). Add `--load-in-4bit` if memory is tight.

In [ ]:
!python scripts/generate.py --base-model {BASE_MODEL} --adapter {SMOKE_DIR}/final \
    --dataset {HF_DATASET} --split test --output outputs/preds_smoke.jsonl --batch-size 16
!python scripts/evaluate.py --predictions outputs/preds_smoke.jsonl --report outputs/report_smoke.json

## Compare

In [ ]:
import json
for name in ["base", "smoke"]:
    path = f"outputs/report_{name}.json"
    if not os.path.exists(path):
        continue
    r = json.load(open(path))
    groups = {"overall": r, **r["by_table_count"]}
    print(name.upper())
    for g, m in groups.items():
        print(f"  {g:<13} n={m['n']:<4} valid_sql={m['valid_sql_pct']}%  exact_match={m['exact_match_pct']}%")

## Download predictions

In [ ]:
from google.colab import files
for f in ["outputs/preds_base.jsonl", "outputs/preds_smoke.jsonl",
          "outputs/report_base.json", "outputs/report_smoke.json"]:
    if os.path.exists(f):
        files.download(f)
    else:
        print("missing:", f)